In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from hydra import compose, initialize

from main.model.script.hydra_beans import KdConfig

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

To evaluate the good contribution of modalities we see MRR of it with and without the metric <br>
So if I have EEG, Aud, Txt and Vid and decide to ablate *vid* I measure:

A:MRR_mean modalities model that can use Vid but without video in inputstream <br>
B:MRR_mean across modalities of the ablated model

If B > A → Video is likely hurting other modalities<br>
If B < A → Video is likely helping them<br>
If B ~ A → Video has little effect on them<br>

In [ ]:
from main.model.neegavi.factory import Factory
from main.model.neegavi.utils import get_model_ckpt

# TODO change?
baseline_checkpoint_path = ""
ckpt = get_model_ckpt(baseline_checkpoint_path)
baseline = Factory.best_inference().build()
baseline.load_state_dict(ckpt)
baseline.eval()

In [ ]:
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule

trainer = lightning.Trainer(precision="16-mixed")

# Audio

In [1]:
from main.core_data.media.audio import Audio

audio_less_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-aud/2026-03-22_22-14-48/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set(Audio.modality_code())).build()
mod_less_model.load_state_dict(inference_ckpt)
mod_less_model.eval()

Run evaluations on data now

In [4]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except audio
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

baseline_results = trainer.validate(baseline, datamodule=datamodule)
audio_less_results = trainer.validate(mod_less_model, datamodule=datamodule)

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'train-local.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)


In [ ]:
baseline_results

In [ ]:
audio_less_results

# Txt

In [ ]:
from main.core_data.media.text import Text

audio_less_checkpoint_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-22_11-24-33/checkpoints/epochepoch=38-stepstep=99567.ckpt"

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set(Text.modality_code())).build()
mod_less_model.load_state_dict(inference_ckpt)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except audio
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

baseline_results = trainer.validate(baseline, datamodule=datamodule)
audio_less_results = trainer.validate(mod_less_model, datamodule=datamodule)

In [ ]:
baseline_results

In [ ]:
audio_less_results

# ECG

In [ ]:
from main.core_data.media.text import Text

audio_less_checkpoint_path = ""

inference_ckpt = get_model_ckpt(audio_less_checkpoint_path)
mod_less_model = Factory.best_inference(disabled_supports=set(Text.modality_code())).build()
mod_less_model.load_state_dict(inference_ckpt)
mod_less_model.eval()

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except audio
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

baseline_results = trainer.validate(baseline, datamodule=datamodule)
audio_less_results = trainer.validate(mod_less_model, datamodule=datamodule)

In [ ]:
baseline_results

In [ ]:
audio_less_results